## Install dependencies

In [ ]:
!pip install -r requirements.txt

In [ ]:
%%sh 
sudo apt-get update
sudo apt-get install -y \
    build-essential \
    pkg-config \
    libudev-dev llvm libclang-dev \
    protobuf-compiler libssl-dev
curl --proto '=https' --tlsv1.2 -sSfL https://solana-install.solana.workers.dev | bash

In [ ]:
import os
# Add Solana to PATH
os.environ['PATH'] = f"/home/sagemaker-user/.local/share/solana/install/active_release/bin:{os.environ['PATH']}"
# Add Cargo to PATH
CARGO_PATH = f"{os.environ['HOME']}/.cargo/bin"
os.environ['PATH'] = f"{CARGO_PATH}:{os.environ['PATH']}"

# Autonomous AI Advertising Agent with Crypto Payments

## ⚠️ **SAFETY WARNINGS - READ BEFORE PROCEEDING**

### 🚨 **TESTNET ONLY - DO NOT USE REAL FUNDS**
- **This notebook uses BASE SEPOLIA TESTNET** - never use mainnet
- **Use only TEST TOKENS** - do not fund with real cryptocurrency
- **Wallet credentials are for development only** - never use production keys
- **Educational demonstration** - not intended for production use

### 🔒 **Security Best Practices**
- ✅ Use separate test credentials and API keys
- ✅ Verify you're on Base Sepolia testnet (not mainnet)
- ✅ Only use testnet USDC tokens (no real value)
- ❌ Never expose real wallet private keys
- ❌ Never run this code with production credentials
- ❌ Never use mainnet addresses or real cryptocurrency

---
## Overview

This notebook demonstrates how to build an **autonomous AI advertising agent** that can create complete marketing campaigns while automatically paying for premium services using cryptocurrency. By combining cutting-edge AI agent frameworks with blockchain technology, we create a system that operates independently in the digital economy.

## What This Notebook Demonstrates

### 🤖 **Autonomous Agent Architecture**
- Built using the **Strands AI framework** with Claude Sonnet 4
- Integrated with **Coinbase AgentKit** for blockchain functionality
- Capable of making independent decisions and executing complex workflows

### 💳 **Crypto-Native Payments**
- **X402 Payment Protocol**: Automatic micropayments for API services
- **Base Sepolia Network**: EVM-compatible blockchain transactions
- **USDC Payments**: Stable currency for predictable service costs
- **Autonomous Wallet Management**: No human intervention required

### 🎨 **Premium Service Integration**
- **AI Image Generation**: Creates custom visuals for ad campaigns
- **Weather Data APIs**: Enables location and season-based targeting
- **Payment-Gated Access**: Demonstrates real economic interactions

### 📊 **Complete Campaign Creation**
- **Multi-Platform Ads**: Social media, display, email, and print formats
- **Dynamic Content**: Adapts messaging based on real-time data
- **Visual Integration**: Embeds generated images into campaign materials
- **HTML Output**: Produces ready-to-deploy advertising assets

## Key Innovation: Economic Agency

This agent represents a new paradigm where AI systems can:
- 🏦 **Manage their own budgets** for operational expenses
- 🛒 **Purchase services** needed to complete tasks
- 🔄 **Optimize spending** based on campaign requirements
- 📈 **Scale operations** without human financial oversight

## Real-World Applications

- **Marketing Agencies**: Automated campaign generation and optimization
- **E-commerce**: Dynamic product advertising with seasonal adaptations
- **Local Businesses**: Location-based campaigns with weather integration
- **Content Creators**: Automated visual and copy generation workflows

## Technical Architecture

The notebook showcases integration between:
- **Strands AI Framework** → Agent reasoning and tool orchestration
- **Coinbase AgentKit** → Blockchain wallet and transaction management
- **X402 Protocol** → HTTP-based micropayment standard
- **External APIs** → Premium image generation and weather services

---

**Run through this notebook to see how autonomous agents can operate as independent economic actors in the digital advertising ecosystem!**

In [ ]:
import json
import sys
import time
import os
import botocore.exceptions
import re
from typing import Dict, Any, Optional, List
from strands import Agent, tool
from strands.models import BedrockModel
from strands_tools import image_reader
import base64
from pathlib import Path
from dotenv import load_dotenv

from solana_x402_client import (
    SolanaX402Client,
    load_keypair_from_env,
    decode_x_payment_response,
)
import subprocess

In [ ]:
# Load environment variables
load_dotenv(dotenv_path="./paid-endpoint-server/.env")
# Replace with your CDP test wallet. If env variables not already set, uncomment and set.

# Service Discovery System and Custom Tools

This cell creates a service catalog and custom tools for external API integration:

## Service Catalog
Defines available external services, that matches the paid service endpoints, with complete schemas:

### 🎨 Media Creation Service
- **Purpose**: AI image generation for ad visuals
- **Cost**: Payment required via X402 protocol
- **Input**: Text prompts describing desired images
- **Output**: Generated image URLs/paths

### 🌤️ Weather Service  
- **Purpose**: Real-time weather data for location-based campaigns
- **Cost**: Payment required via X402 protocol
- **Input**: City name and unit preferences
- **Output**: Current weather conditions and forecasts

## Custom Agent Tools

### `@tool list_available_services()`
- Lists all available external services
- Helps agent discover capabilities dynamically

### `@tool get_service_schema(service_name)`
- Returns detailed API schema for specific services
- Enables proper request formatting

### `@tool create_ad_html(ad_html_content, file_name, output_path)`
- Generates HTML files containing complete ad campaigns
- Embeds images and formats content for web display
- Creates organized file structure for campaign assets

These tools enable intelligent service discovery and seamless integration with payment-gated APIs.

In [ ]:
# Configuration
PORT = os.getenv("PORT", 4034)
BASE_URL = os.getenv("BASE_URL", f"http://0.0.0.0:{PORT}") # Replace port number with actual service port if modified

# Service Discovery Functions
def external_service_catalog(service_name: str) -> Optional[Dict[str, Any]]:
    """
    Returns service endpoint information including schemas and URLs.

    Args:
        service_name: Name of the service to query

    Returns:
        Dictionary containing service metadata, schemas, and endpoint URL
    """
    catalog = {
        "media-creation": {
            "name": "Image Generator",
            "description": "Generate images based on text prompts using AI. Perfect for creating ad visuals, product shots, and marketing materials.",
            "url": f"{BASE_URL}/image_generator",
            "method": "POST",
            "payment_required": True,
            "input_schema": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": ["string", "array"],
                        "description": "The prompt(s) describing the image to generate. Can be a single string (e.g., 'Generate image of a sunset') or an array of strings for different prompts (e.g., ['sunset over mountains', 'city at night'])",
                        "minLength": 1,
                        "maxLength": 500,
                        "minItems": 1,
                        "maxItems": 4
                    }
                },
                "required": ["query"]
            }
        },

        "weather": {
            "name": "Weather Information",
            "description": "Get current weather information for any city. Useful for location-based or seasonal ad campaigns.",
            "url": f"{BASE_URL}/weather",
            "method": "POST",
            "payment_required": True,
            "input_schema": {
                "type": "object",
                "properties": {
                    "city": {
                        "type": "string",
                        "description": "City name (e.g., 'London' or 'London,UK')",
                        "example": "London,UK"
                    },
                    "units": {
                        "type": "string",
                        "description": "Units: 'metric', 'imperial', or 'standard'",
                        "default": "metric"
                    }
                },
                "required": ["city"]
            }
        }
    }

    return catalog.get(service_name)


# Custom Tools for the Agent
@tool
def list_available_services() -> str:
    """
    List all available external services that can be used for ad creation.
    Returns information about available endpoints including weather data and image generation.
    Use this when you need to know what external services are available for your ad campaign.
    """
    services = {
        "media-creation": external_service_catalog("media-creation"),
        "weather": external_service_catalog("weather")
    }

    result = "Available Services for Content Creation:\n\n"

    for service_name, service_info in services.items():
        if service_info:
            result += f"Service: {service_info['name']}\n"
            result += f"  Description: {service_info['description']}\n"
    return result


@tool
def get_service_schema(service_name: str) -> str:
    """
    Get detailed schema information and URL endpoint for a specific service.

    Args:
        service_name: Name of the service ('media-creation' or 'weather')

    Returns:
        Detailed schema information including input requirements
    """
    service = external_service_catalog(service_name)

    if not service:
        return f"Service '{service_name}' not found. Available services: 'media-creation', 'weather'"

    result = f"Service: {service['name']}\n"
    result += f"Description: {service['description']}\n\n"
    result += f"Endpoint: {service['method']} {service['url']}\n\n"
    result += f"Input Schema:\n{json.dumps(service['input_schema'], indent=2)}\n"
    
    return result


@tool
def create_ad_html(
    ad_html_content: str,
    file_name: str,
    output_path: str = "ad_campaign",
) -> str:
    """
    Create an HTML file displaying the ad campaign.

    Args:
        ad_html_content: Html data containing ad content including generated images (if used) with html syntax used in creating the html file
        file_name: name of the html file including the .html extension
        output_path: Path where HTML file will be saved

    Returns:
        Path to the created HTML file
    """
    # Write to file
    output_file = Path(f"{output_path}/{file_name}")
    output_file.parent.mkdir(parents=True, exist_ok=True)

    with open(output_file, 'w', encoding='utf-8') as f:
        f.write(ad_html_content)

    return f"HTML file saved to {str(output_file.absolute())}"


## SOLANA BLOCKCHAIN ACTIONS

In [ ]:
@tool
def check_account_balance() -> Dict[str, Any]:
    """
    Check account balance for the default configured wallet.
    Returns SOL balance and all token balances.
    """
    try:
        # Get wallet address
        address_result = subprocess.run(
            ["solana", "address"],
            capture_output=True,
            text=True,
            check=True
        )
        wallet_address = address_result.stdout.strip()

        # Get SOL balance
        sol_result = subprocess.run(
            ["solana", "balance"],
            capture_output=True,
            text=True,
            check=True
        )
        sol_balance_str = sol_result.stdout.strip().split()[0]
        sol_balance = float(sol_balance_str)

        # Get all token balances
        token_result = subprocess.run(
            ["spl-token", "accounts", "--output", "json"],
            capture_output=True,
            text=True,
            check=True
        )

        token_data = json.loads(token_result.stdout)
        tokens = []

        for account in token_data.get("accounts", []):
            mint = account.get("mint")
            token_amount = account.get("tokenAmount", {})

            # Extract from tokenAmount object
            amount = token_amount.get("amount")  # Atomic amount as string
            decimals = token_amount.get("decimals", 6)
            ui_amount = token_amount.get("uiAmount")  # Display amount as float

            if mint and amount is not None:
                balance = ui_amount if ui_amount is not None else (float(amount) / (10 ** decimals))
                atomic_balance = int(amount)

                tokens.append({
                    "mint": mint,
                    "balance": balance,
                    "atomic_balance": atomic_balance,
                    "decimals": decimals,
                    # "account_address": account.get("address"),
                    # "is_associated": account.get("isAssociated", False),
                    # "program_id": account.get("programId")
                })

        return {
            "wallet_address": wallet_address,
            "sol_balance": sol_balance,
            "tokens": tokens
        }

    except subprocess.CalledProcessError as e:
        raise Exception(f"Failed to check balance: {e.stderr or str(e)}")
    except Exception as e:
        raise Exception(f"Error checking balance: {str(e)}")


@tool
def request_faucet_tokens(asset_id: str, amount: float = 1000) -> Dict[str, Any]:
    """
    Request tokens from faucet or mint tokens.
    For SOL, requests airdrop on devnet.
    For SPL tokens, mints tokens (requires mint authority).

    Args:
        asset_id: Token mint address or "SOL" for native SOL
        amount: Amount to request (in token units, not atomic)

    Returns:
        Dictionary with transaction details

    Example:
        >>> # Request SOL airdrop
        >>> result = request_faucet_tokens("SOL", amount=2)
        >>> print(result['message'])

        >>> # Mint custom tokens
        >>> result = request_faucet_tokens("CvArN2zZbDkswQ4g8MZZUNnkJHbHpM5j7isaLfMARGbD", amount=1000)
        >>> print(result['message'])
    """
    try:
        # Special case: SOL airdrop
        if asset_id.upper() == "SOL":
            result = subprocess.run(
                ["solana", "airdrop", str(amount)],
                capture_output=True,
                text=True,
                check=True
            )

            # Extract signature
            signature = None
            for line in result.stdout.split("\n"):
                if "Signature:" in line:
                    signature = line.split("Signature:")[1].strip()
                    break

            return {
                "success": True,
                "asset": "SOL",
                "amount": amount,
                "signature": signature,
                "message": f"✓ Received {amount} SOL airdrop"
            }

        # For SPL tokens, mint tokens
        else:
            # First, ensure token account exists
            subprocess.run(
                ["spl-token", "create-account", asset_id],
                capture_output=True,
                text=True
            )
            # Ignore error if account already exists

            # Mint tokens
            result = subprocess.run(
                ["spl-token", "mint", asset_id, str(amount)],
                capture_output=True,
                text=True,
                check=True
            )

            # Extract signature
            signature = None
            for line in result.stdout.split("\n"):
                if "Signature:" in line:
                    signature = line.split("Signature:")[1].strip()
                    break

            return {
                "success": True,
                "asset": asset_id,
                "amount": amount,
                "signature": signature,
                "message": f"✓ Minted {amount} tokens"
            }

    except subprocess.CalledProcessError as e:
        error_msg = e.stderr or str(e)

        if "Invalid mint authority" in error_msg or "missing required signature" in error_msg:
            return {
                "success": False,
                "asset": asset_id,
                "amount": amount,
                "error": "mint_authority_required",
                "message": (
                    f"✗ Failed to mint tokens: You don't have mint authority.\n"
                    f"  This token is owned by someone else.\n"
                    f"  You need to request tokens from the token owner."
                )
            }

        return {
            "success": False,
            "asset": asset_id,
            "amount": amount,
            "error": error_msg,
            "message": f"✗ Failed to request tokens: {error_msg}"
        }
    except Exception as e:
        return {
            "success": False,
            "asset": asset_id,
            "amount": amount,
            "error": str(e),
            "message": f"✗ Error: {str(e)}"
        }


def strip_ansi_codes(text: str) -> str:
    """
    Remove ANSI escape codes from text.

    Args:
        text: Text containing ANSI codes

    Returns:
        Clean text without ANSI codes
    """
    ansi_escape = re.compile(r'\x1b\[[0-9;]*m')
    return ansi_escape.sub('', text)

@tool
def get_wallet_details() -> Dict[str, Any]:
    """
    Get wallet details including wallet address, network, RPC URL, and balances.

    Returns:
        Dictionary containing wallet details

    Example:
        >>> details = get_wallet_details()
        >>> print(f"Address: {details['address']}")
        >>> print(f"Network: {details['network']}")
        >>> print(f"SOL Balance: {details['sol_balance']}")
        >>> print(f"RPC URL: {details['rpc_url']}")
    """
    try:
        # Get wallet address
        address_result = subprocess.run(
            ["solana", "address"],
            capture_output=True,
            text=True,
            check=True
        )
        wallet_address = address_result.stdout.strip()

        # Get SOL balance
        balance_result = subprocess.run(
            ["solana", "balance"],
            capture_output=True,
            text=True,
            check=True
        )
        sol_balance_str = balance_result.stdout.strip().split()[0]
        sol_balance = float(sol_balance_str)

        # Get configuration (network, RPC URL, etc.)
        config_result = subprocess.run(
            ["solana", "config", "get"],
            capture_output=True,
            text=True,
            check=True
        )

        # Strip ANSI escape codes
        clean_output = strip_ansi_codes(config_result.stdout)

        # Parse config
        config_lines = clean_output.strip().split("\n")
        config_data = {}

        for line in config_lines:
            if ":" in line:
                # Split only on first colon
                key, value = line.split(":", 1)
                # Clean up the value (remove extra info like "(computed)")
                value_clean = value.strip()
                # Remove parenthetical notes like "(computed)"
                if "(" in value_clean:
                    value_clean = value_clean.split("(")[0].strip()

                config_data[key.strip()] = value_clean

        config_file = config_data.get("Config File", "")
        rpc_url = config_data.get("RPC URL", "")
        ws_url = config_data.get("WebSocket URL", "")
        keypair_path = config_data.get("Keypair Path", "")
        commitment = config_data.get("Commitment", "")

        # Determine network from RPC URL
        if "devnet" in rpc_url.lower():
            network = "devnet"
        elif "testnet" in rpc_url.lower():
            network = "testnet"
        elif "mainnet" in rpc_url.lower():
            network = "mainnet-beta"
        else:
            network = "unknown"

        # Get token count
        try:
            token_result = subprocess.run(
                ["spl-token", "accounts", "--output", "json"],
                capture_output=True,
                text=True,
                check=True
            )
            token_data = json.loads(token_result.stdout)
            token_count = len(token_data.get("accounts", []))
        except:
            token_count = 0

        return {
            "address": wallet_address,
            "network": network,
            # "config_file": config_file,
            "rpc_url": rpc_url,
            # "websocket_url": ws_url,
            # "keypair_path": keypair_path,
            "commitment": commitment,
            "sol_balance": sol_balance,
            "token_account_count": token_count
        }

    except subprocess.CalledProcessError as e:
        raise Exception(f"Failed to get wallet details: {e.stderr or str(e)}")
    except Exception as e:
        raise Exception(f"Error getting wallet details: {str(e)}")


@tool
def get_token_balance(asset_id: str, owner: Optional[str] = None) -> Dict[str, Any]:
    """
    Get balance for a specific token.

    Args:
        asset_id: Token mint address
        owner: Optional owner address. If None, uses default wallet.

    Returns:
        Dictionary with token balance information

    Example:
        >>> balance = get_token_balance("CvArN2zZbDkswQ4g8MZZUNnkJHbHpM5j7isaLfMARGbD")
        >>> print(f"Balance: {balance['balance']}")
    """
    try:
        cmd = ["spl-token", "balance", asset_id]
        if owner:
            cmd.extend(["--owner", owner])

        result = subprocess.run(
            cmd,
            capture_output=True,
            text=True,
            check=True
        )

        balance_str = result.stdout.strip()
        balance = float(balance_str)

        # Assume 6 decimals (standard for most tokens)
        decimals = 6
        atomic_balance = int(balance * (10 ** decimals))

        return {
            "mint": asset_id,
            "balance": balance,
            "atomic_balance": atomic_balance,
            "decimals": decimals,
            # "owner": owner or "default wallet"
        }

    except subprocess.CalledProcessError as e:
        error_msg = e.stderr or str(e)
        if "Could not find" in error_msg or "not found" in error_msg:
            return {
                "mint": asset_id,
                "balance": 0,
                "atomic_balance": 0,
                "decimals": 6,
                # "owner": owner or "default wallet",
                "error": "Account not found - balance is 0"
            }
        raise Exception(f"Failed to get token balance: {error_msg}")
    except Exception as e:
        raise Exception(f"Error getting token balance: {str(e)}")


## x402 Requests Tools

In [ ]:
# Create x402 client
keypair = load_keypair_from_env("SOLANA_PRIVATE_KEY")
client = SolanaX402Client(
    keypair=keypair,
    rpc_url="https://api.devnet.solana.com",
)

solana_client_tools = [
                client.make_request_with_payment,
                client.make_request_
            ]

# Advertising Agent Configuration

This cell sets up our specialized advertising agent with comprehensive capabilities:

## Model Configuration (Optional)
The commented BedrockModel shows advanced configuration options:
- **Claude Sonnet 4**: Latest high-performance reasoning model
- **Thinking Budget**: 2048 tokens for internal reasoning
- **Cache Settings**: Optimized for prompt and tool caching

## Tool Integration
Combines multiple tool categories:
- **Custom Tools**: Service discovery and HTML generation
- **Strands Tools**: Image reading and processing
- **X402 Tools**: Payment-gated API access

## System Prompt Design
The comprehensive system prompt defines the agent as an expert in:

### 🎯 Core Capabilities
- Digital marketing and copywriting
- Multi-platform ad creation (social, display, email, print)
- Visual content generation and integration
- Campaign strategy and optimization

### 🛠️ Technical Workflow
1. **Discovery**: Use service catalog to find relevant APIs
2. **Enhancement**: Generate visuals and gather contextual data
3. **Creation**: Craft compelling copy with clear CTAs
4. **Integration**: Combine all elements into cohesive campaigns
5. **Output**: Generate complete HTML files with embedded media

### ✅ Quality Standards
- Mobile-first design approach
- A/B testing recommendations
- Proper image embedding in HTML
- Brand consistency guidelines
- Performance optimization

This agent is designed to create production-ready advertising campaigns autonomously.

In [ ]:
# Initialize the Bedrock Model

# bedrock_model = BedrockModel(
#     cache_prompt="default",
#     cache_tools="default",
#     model_id="us.anthropic.claude-sonnet-4-20250514-v1:0",
#     max_tokens=4000,
#     additional_request_fields={
#         "thinking": {
#             "type": "enabled",
#             "budget_tokens": 2048 # Minimum of 1,024
#         }
#     }
# )


# Create the Ads Creative Agent
advertising_agent = Agent(
    # model=bedrock_model,  # defaults to Sonnet 4
    tools=[
        list_available_services,
        get_service_schema,
        image_reader,
        create_ad_html,
    ] + solana_client_tools,
    system_prompt="""You are an expert advertising and marketing content creator with extensive experience in digital marketing, copywriting, and creative direction.

Your capabilities:
1. Create compelling ad copy for various platforms (social media, display, email, print)
2. Discover and use external services for ad enhancement:
   - Generate professional visuals using the image generation service
   - Get real-time weather data for location-based and seasonal campaigns
3. Design complete ad campaigns with copy, visuals, and targeting strategy
4. Interacting with on chain actions for making and necessary payments
5. Tool for creating a html file of the generated ads

When creating ads:
- First start with and outline of the ads to be created
- Use list_available_services() to discover external services that can help in creating ads
- Consider the target audience, platform, and campaign goals
- If location-specific or weather-dependent, use the appropiate external service to get weather info
- Always generate relevant visuals using image endpoint service
- Use persuasive copywriting techniques (AIDA, PAS, FAB)
- Include clear call-to-action (CTA)
- Optimize for the specific ad format and platform
- Save the ads content as a html file. When creating the html content, make sure images are rendered properly. Take this part very seriously

Ad Formats You Can Create:
- Social Media Ads (Facebook, Instagram, LinkedIn, Twitter)
- Display Banner Ads
- Email Marketing
- Print Advertisements
- Video Ad Scripts
- Search Ads (Google, Bing)

Response Structure:
1. Campaign Overview (objective, audience, platform)
2. Headline(s) - attention-grabbing and benefit-focused
3. Body Copy - persuasive and engaging
4. Call-to-Action - clear and action-oriented
5. Visual Description/Generated Images
6. Targeting Recommendations (demographics, interests, behaviors)
7. Budget & Timing Suggestions (if applicable)
8. **HTML File Creation Checklist:**
   - [ ] All images generated and URLs obtained
   - [ ] Images embedded in HTML with <img> tags using relative paths
   - [ ] HTML file saved with rendered images

Best Practices:
- Keep it concise and scannable
- Focus on benefits, not just features
- Use emotional triggers and power words
- Create urgency when appropriate
- Ensure brand consistency
- Mobile-first approach
- A/B testing recommendations
- Include images in teh generated html file

CRITICAL REQUIREMENT - HTML FILE GENERATION:
When saving ads as HTML files, you MUST:
✓ Include ALL generated images embedded in the HTML
✓ Use proper <img> tags with src attributes pointing to the image URLs
✓ Ensure images are visible when the HTML file is opened
✓ Test that image URLs are accessible and properly formatted
✓ Never create HTML files with missing or broken image references
**NEVER use absolute file system paths:**
❌ `<img src="/home/user/file-path/output/image.png"...>`

**ALWAYS use relative paths based on the HTML file location:**
✅ `<img src="../output/image.png"...>`


BEFORE saving the HTML file, verify:
1. Each ad section has its corresponding image embedded
2. Image URLs are complete and valid
3. Images have appropriate alt text
4. The HTML renders images correctly

Be creative, data-driven, and results-oriented!"""
)

In [ ]:
response = advertising_agent(""" Generate an ad for the following:
    product="ice cream shop promotion",
    city="Lagos,NG",
    platform="Instagram"
""")

You can validate transactions after getting the hash at  `https://solscan.io/?cluster=devnet`

In [ ]:
response = advertising_agent("""What are the transactions hashes and how can I validate them
""")